# Class Activity 3 — Make the embeddings (VS Code version)

This notebook turns every title description into a vector of **384 numbers** using the free model **all-MiniLM-L6-v2**, then writes two SQL files that load the vectors into PostgreSQL. It runs on your own computer, in VS Code.

**Before you start** (the guide *Making the Embeddings*, Step 2, Route B, has the details)
1. Keep this notebook, **`ca3_titles.csv`** and **`requirements.txt`** in the same folder, and open that folder in VS Code (**File → Open Folder…**).
2. Create the environment: **Python: Create Environment** → **Venv** → Python 3.12 → tick **requirements.txt**.
3. Click **Select Kernel** (top right) → **Python Environments…** → **.venv**. (Any environment that already has sentence-transformers, such as your Week 9 one, also works.)
4. If VS Code asks to install **ipykernel**, click **Install**.
5. Click **Run All** at the top of the notebook, or run the cells one at a time, top to bottom.

The two SQL files are saved in a **`my_sql`** folder next to this notebook.

## Cell 1 — check the libraries and load the model

If the libraries are missing from the Python you picked, this cell installs them (`sentence-transformers` and `pandas`, up to about 2 GB). Picking the `.venv` kernel you created with `requirements.txt` avoids that.

The first run downloads the model (about 90 MB). Later runs load it from your computer in a few seconds.

**Expect, at the end:** `Model ready: all-MiniLM-L6-v2, 384 dimensions.`

In [ ]:
import sys
print("Python:", sys.executable)
assert sys.version_info >= (3, 10), "This needs Python 3.10 or newer. Pick another kernel (top right)."

try:
    import pandas as pd
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError as e:
    print(f"'{e.name}' is not installed in this Python, so installing sentence-transformers and pandas now.")
    print("This can take several minutes. To skip it, pick the .venv kernel you created with requirements.txt.")
    get_ipython().run_line_magic("pip", "install -q sentence-transformers pandas")
    import pandas as pd
    from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"   # the version the course uses
try:
    model = SentenceTransformer(MODEL_NAME, revision=REVISION)
except Exception:
    try:
        model = SentenceTransformer(MODEL_NAME)
        print("Could not load the course's model version, so loaded the latest one instead.")
    except Exception as e:
        raise SystemExit(
            "The model could not be loaded.\n"
            f"  Error: {type(e).__name__}: {str(e)[:300]}\n"
            "  The first download needs huggingface.co, which some campus networks block:\n"
            "  try home Wi-Fi or a phone hotspot. On a Mac, if the error mentions certificates,\n"
            "  run 'Install Certificates.command' in Applications > Python 3.x.")

get_dim = getattr(model, "get_embedding_dimension", None) or model.get_sentence_embedding_dimension
DIM = get_dim()
assert DIM == 384, f"Expected 384 dimensions, got {DIM}. Wrong model?"
print(f"Model ready: all-MiniLM-L6-v2, {DIM} dimensions.")

## Cell 2 — read the titles

Reads **`ca3_titles.csv`** from this notebook's folder.

Using your own catalogue instead? Save it in this folder as `ca3_titles.csv`, with these six columns: `title_id, title, kind, release_year, genre, description`.

**Expect:** `Loaded 60 titles.` (or the number of rows in your own file)

In [ ]:
from pathlib import Path

HERE = Path.cwd()
CSV = HERE / "ca3_titles.csv"
assert CSV.exists(), (f"ca3_titles.csv is not in {HERE}.\n"
                      "Keep it in the same folder as this notebook, and open that folder in VS Code.")
titles = pd.read_csv(CSV)

need = ["title_id", "title", "kind", "release_year", "genre", "description"]
missing = [c for c in need if c not in titles.columns]
assert not missing, f"ca3_titles.csv is missing these columns: {missing}"
assert titles["title_id"].is_unique, "Two rows share the same title_id. Each title needs its own id."
titles["description"] = titles["description"].fillna("").astype(str).str.strip()
assert (titles["description"] != "").all(), "Some titles have an empty description. Fill them in and run this cell again."
print(f"Loaded {len(titles)} titles.")

## Cell 3 — make the title vectors and write the SQL file

Each description goes into the model and comes out as 384 numbers. Titles with similar meanings get similar numbers.

**Expect:**
- `60 vectors × 384 numbers, every one unit length: True`
- the five titles closest in meaning to title 1, as a quick check
- `Wrote …/my_sql/ca3_titles_vectors.sql`

In [ ]:
import numpy as np
from datetime import date

OUT = HERE / "my_sql"
OUT.mkdir(exist_ok=True)

title_vecs = np.asarray(model.encode(titles["description"].tolist(), normalize_embeddings=True,
                                     show_progress_bar=True), dtype="float32")
ok = bool(np.allclose(np.linalg.norm(title_vecs, axis=1), 1, atol=1e-4))
print(f"{title_vecs.shape[0]} vectors × {title_vecs.shape[1]} numbers, every one unit length: {ok}")

# Quick check. Cosine distance = 1 - dot product (pgvector's <=> operator). 0 = same meaning.
d = 1 - title_vecs @ title_vecs[0]
print(f"\nClosest in meaning to: {titles['title'].iloc[0]}")
for i in np.argsort(d, kind="stable")[1:6]:
    print(f"  {d[i]:.3f}  {titles['title'].iloc[i]}  ({titles['genre'].iloc[i]})")

def vec(v):
    return "[" + ",".join(f"{x:.6f}".rstrip("0").rstrip(".") for x in v) + "]"

def q(s):
    return "'" + str(s).replace("'", "''") + "'"

def num(x):
    return "NULL" if pd.isna(x) else str(int(x))

rows = [f"({int(r.title_id)},{q(r.title)},{q(r.kind)},{num(r.release_year)},{q(r.genre)},"
        f"{q(r.description)},{q(vec(title_vecs[i]))})"
        for i, r in enumerate(titles.itertuples(index=False))]

sql = f"""-- Class Activity 3 · title vectors
-- Made in VS Code on {date.today().isoformat()} with all-MiniLM-L6-v2 (384 dimensions).
-- Run this WHOLE file in pgAdmin or VS Code, connected to your team's database.
-- Safe to run again: existing titles are updated, not duplicated.

CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS titles (
    title_id     integer PRIMARY KEY,
    title        text    NOT NULL,
    kind         text    NOT NULL,
    release_year integer,
    genre        text,
    description  text    NOT NULL,
    embedding    vector(384)
);

INSERT INTO titles (title_id, title, kind, release_year, genre, description, embedding) VALUES
""" + ",\n".join(rows) + """
ON CONFLICT (title_id) DO UPDATE SET
    title = EXCLUDED.title, kind = EXCLUDED.kind, release_year = EXCLUDED.release_year,
    genre = EXCLUDED.genre, description = EXCLUDED.description, embedding = EXCLUDED.embedding;

-- Vector index for cosine distance (the <=> operator).
CREATE INDEX IF NOT EXISTS titles_embedding_idx ON titles USING hnsw (embedding vector_cosine_ops);

-- Self-check. Expect: titles = with_vectors, and dimensions = 384.
SELECT count(*) AS titles, count(embedding) AS with_vectors, max(vector_dims(embedding)) AS dimensions
FROM titles;
"""

(OUT / "ca3_titles_vectors.sql").write_text(sql, encoding="utf-8")
print(f"\nWrote {OUT / 'ca3_titles_vectors.sql'}")

## Cell 4 — your search phrases

PostgreSQL can compare vectors, but it **cannot turn text into vectors** by itself. So any phrase you want to search with must go through the same model here first.

Edit the list below: keep each phrase in quotes, with a comma after it. Then run the cell.

**Expect:** the three closest titles for each phrase, then `Wrote …/my_sql/ca3_search_phrases.sql`.

Changed your phrases later? Edit the list, run this cell again, and run the new file again in your SQL editor.

In [ ]:
MY_PHRASES = [
    "a cozy mystery in a small town",
    "surviving alone in space",
    "a cooking competition",
    "something scary in the woods",
    "a gentle cartoon for young kids",
    # Group 5's own phrases:
    "a true story about the ocean",
    "a heist that goes hilariously wrong",
]

phrase_vecs = np.asarray(model.encode(MY_PHRASES, normalize_embeddings=True), dtype="float32")
for n, (phrase, v) in enumerate(zip(MY_PHRASES, phrase_vecs), start=1):
    d = 1 - title_vecs @ v
    print(f"\n{n}. {phrase}")
    for i in np.argsort(d, kind="stable")[:3]:
        print(f"   {d[i]:.3f}  {titles['title'].iloc[i]}  ({titles['genre'].iloc[i]})")

rows = [f"({n},{q(p)},{q(vec(v))})" for n, (p, v) in enumerate(zip(MY_PHRASES, phrase_vecs), start=1)]
sql = f"""-- Class Activity 3 · your search phrases
-- Made in VS Code on {date.today().isoformat()} with all-MiniLM-L6-v2 (384 dimensions).
-- Run this WHOLE file in pgAdmin or VS Code. It replaces any phrases you loaded before.

CREATE EXTENSION IF NOT EXISTS vector;

DROP TABLE IF EXISTS search_phrases;
CREATE TABLE search_phrases (
    phrase_id integer PRIMARY KEY,
    phrase    text        NOT NULL,
    embedding vector(384) NOT NULL
);

INSERT INTO search_phrases (phrase_id, phrase, embedding) VALUES
""" + ",\n".join(rows) + """;

SELECT phrase_id, phrase FROM search_phrases ORDER BY phrase_id;
"""

(OUT / "ca3_search_phrases.sql").write_text(sql, encoding="utf-8")
print(f"\nWrote {OUT / 'ca3_search_phrases.sql'}")

## Done here. Next: load the files into your database

Open the **`my_sql`** folder in VS Code's Explorer. Connected to your database, run each file as a **whole file**, in this order:

1. `ca3_titles_vectors.sql` — the last result shows `60 | 60 | 384`
2. `ca3_search_phrases.sql` — the last result lists your phrases

Then check that it worked. This finds the five titles closest in meaning to your first phrase:

```sql
SELECT t.title, t.genre,
       round((t.embedding <=> p.embedding)::numeric, 3) AS distance
FROM titles t, search_phrases p
WHERE p.phrase_id = 1
ORDER BY t.embedding <=> p.embedding
LIMIT 5;
```

The `distance` values should match the ones Cell 4 printed for phrase 1.